# Simulación: Cobardía vs. Altruismo

Estudio evolutivo de cómo la cobardía y el altruismo compiten en una población.

**Tres experimentos:**
1. Altruismo puro vs. cobardía (sin señal de barba verde).
2. Altruismo vinculado a la barba verde (reconocimiento de parientes).
3. Genes independientes: cuatro fenotipos (barba verde y comportamiento desacoplados).

> Todo el código de simulación está en el paquete `simulation/`. Este notebook
> únicamente ejecuta experimentos y muestra resultados.

In [ ]:
# ── Importaciones ──────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # permite importar desde el workspace

from simulation import *

import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings("ignore")

# Estilo de gráficos
for _style in ["seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot"]:
    try:
        plt.style.use(_style); break
    except OSError:
        pass

matplotlib.rcParams.update({
    "figure.dpi":  120,
    "font.size":   11,
    "axes.titlesize": 13,
})
print("Paquete simulation cargado correctamente.")

---
## Experimento 1 — Altruismo Puro vs. Cobardía

En este experimento **no existe la barba verde**: la única distinción entre
criaturas es su comportamiento.

### Mecánica ante un depredador
- **Cobarde:** huye de inmediato → su compañera muere; él sobrevive con seguridad.
- **Altruista:** avisa a su compañera → ella **siempre** escapa; el altruista escapa
  con probabilidad `P`.
- **Individuo solo en el árbol:** siempre muere, sea cobarde o altruista.

### Predicción teórica
La cobardía debería dominar para `P` bajas, ya que el cobarde garantiza su
propia supervivencia a costa del compañero, mientras que el altruista arriesga
su vida por la de otro.

In [ ]:
# ── Configuración base ─────────────────────────────────────────────────────
cfg_exp1 = SimulationConfig(
    num_trees            = 50,
    num_predator_trees   = 10,
    initial_population   = 100,
    num_days             = 200,
    escape_probability   = 0.5,
    offspring_range      = (1, 2),
    max_population       = 2000,
    num_runs             = 15,
    initial_altruist_ratio = 0.5,
    random_seed          = 42,
)

# ── Ejecución ──────────────────────────────────────────────────────────────
runs_exp1 = run_multiple_simulations(
    config                = cfg_exp1,
    strategy              = Experiment1Strategy(),
    population_factory_fn = PopulationFactory.create_exp1_population,
)

# Resumen rápido
final_day   = runs_exp1[0][-1]
n_days_run  = len(runs_exp1[0])
print(f"Días simulados (ejecución 0): {n_days_run}")
print(f"Población final:  {final_day.total_population:>6}")
print(f"  Altruistas:     {final_day.altruist_count:>6}  ({final_day.altruist_fraction:.1%})")
print(f"  Cobardes:       {final_day.coward_count:>6}  ({final_day.coward_fraction:.1%})")

In [ ]:
plot_population_evolution(
    runs_exp1, cfg_exp1,
    title      = "Experimento 1 — Evolución de la Población",
    show_types = "exp1",
)

plot_altruist_fraction(
    runs_exp1, cfg_exp1,
    title = "Experimento 1 — Fracción de Altruistas a lo largo del Tiempo",
)

In [ ]:
# ── Efecto de P (probabilidad de escape del altruista) ────────────────────
compare_escape_probabilities(
    escape_probs          = [0.1, 0.25, 0.5, 0.6, 0.75, 0.9, 1.0],
    config_base           = cfg_exp1,
    strategy_class        = Experiment1Strategy,
    population_factory_fn = PopulationFactory.create_exp1_population,
    title = "Experimento 1 — Fracción Final de Altruistas vs. Probabilidad de Escape",
)

In [ ]:
# ── Efecto del ratio inicial de altruistas ───────────────────────────────
compare_initial_ratios(
    ratios                = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9],
    config_base           = cfg_exp1,
    strategy_class        = Experiment1Strategy,
    population_factory_fn = PopulationFactory.create_exp1_population,
    title = "Experimento 1 — Efecto del Ratio Inicial de Altruistas",
)

---
## Experimento 2 — Altruismo y la Barba Verde (correlación perfecta)

Se introduce la **barba verde** como señal de reconocimiento de parientes.
En este experimento los genes de altruismo y barba verde están **perfectamente
correlacionados**: todo altruista tiene barba verde y ningún cobarde la tiene.

### Mecánica ante un depredador
- **Cobarde:** huye siempre, sin importar la barba de su compañera.
- **Altruista con compañera de barba verde:** avisa → ella escapa; él escapa con `P`.
- **Altruista con compañera sin barba verde:** actúa como cobarde (¡no avisa!).
- **Individuo solo:** siempre muere.

### Predicción teórica
Como altruista ↔ barba verde, el altruista siempre avisa a otro altruista.
Esto debería favorecer al altruismo frente al Experimento 1.

In [ ]:
cfg_exp2 = SimulationConfig(
    num_trees            = 50,
    num_predator_trees   = 10,
    initial_population   = 100,
    num_days             = 200,
    escape_probability   = 0.5,
    offspring_range      = (1, 2),
    max_population       = 2000,
    num_runs             = 15,
    initial_altruist_ratio = 0.5,
    random_seed          = 42,
)

runs_exp2 = run_multiple_simulations(
    config                = cfg_exp2,
    strategy              = Experiment2Strategy(),
    population_factory_fn = PopulationFactory.create_exp2_population,
)

final_day2 = runs_exp2[0][-1]
print(f"Días simulados (ejecución 0): {len(runs_exp2[0])}")
print(f"Población final:  {final_day2.total_population:>6}")
print(f"  Altruistas (barba verde): {final_day2.altruist_count:>6}  ({final_day2.altruist_fraction:.1%})")
print(f"  Cobardes (sin barba):     {final_day2.coward_count:>6}  ({final_day2.coward_fraction:.1%})")

In [ ]:
plot_population_evolution(
    runs_exp2, cfg_exp2,
    title      = "Experimento 2 — Evolución de la Población",
    show_types = "exp2",
)

plot_altruist_fraction(
    runs_exp2, cfg_exp2,
    title = "Experimento 2 — Fracción de Altruistas a lo largo del Tiempo",
)

In [ ]:
compare_escape_probabilities(
    escape_probs          = [0.1, 0.25, 0.5, 0.6, 0.75, 0.9, 1.0],
    config_base           = cfg_exp2,
    strategy_class        = Experiment2Strategy,
    population_factory_fn = PopulationFactory.create_exp2_population,
    title = "Experimento 2 — Fracción Final de Altruistas vs. Probabilidad de Escape",
)

In [ ]:
compare_initial_ratios(
    ratios                = [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9],
    config_base           = cfg_exp2,
    strategy_class        = Experiment2Strategy,
    population_factory_fn = PopulationFactory.create_exp2_population,
    title = "Experimento 2 — Efecto del Ratio Inicial de Altruistas",
)

In [ ]:
# ── Comparación directa: Exp 1 vs Exp 2 ──────────────────────────────────
import numpy as np

def _final_fracs(runs):
    return [r[-1].altruist_count / r[-1].total_population
            for r in runs if r and r[-1].total_population > 0]

fracs1 = _final_fracs(runs_exp1)
fracs2 = _final_fracs(runs_exp2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([fracs1, fracs2], labels=["Experimento 1\n(sin barba)", "Experimento 2\n(barba verde)"],
           patch_artist=True,
           boxprops=dict(facecolor="#dfe6e9"),
           medianprops=dict(color="#e74c3c", linewidth=2))
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("Fracción Final de Altruistas")
ax.set_title("Comparación Directa — Experimento 1 vs. Experimento 2", fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Exp 1 — media: {np.mean(fracs1):.2%}  std: {np.std(fracs1):.2%}")
print(f"Exp 2 — media: {np.mean(fracs2):.2%}  std: {np.std(fracs2):.2%}")

---
## Experimento 3 — Genes Independientes (cuatro fenotipos)

Se rompe la correlación perfecta del Experimento 2. Los genes de altruismo y
barba verde **segregan de forma independiente**, generando cuatro fenotipos:

| Fenotipo                  | Comportamiento | Señal visual |
|---------------------------|---------------|--------------|
| Altruista + Barba Verde   | Avisa si ve barba verde | Sí |
| Altruista + Sin Barba     | No avisa (como cobarde) | No |
| Cobarde + Barba Verde     | Siempre huye | Sí |
| Cobarde + Sin Barba       | Siempre huye | No |

### Predicción teórica
La barba verde puede ser **explotada** por cobardes que la portan sin ser altruistas.
Esperamos que la señal de barba verde sea inestable y el altruismo tienda a caer.

In [ ]:
cfg_exp3_base = SimulationConfig(
    num_trees              = 50,
    num_predator_trees     = 10,
    initial_population     = 100,
    num_days               = 200,
    escape_probability     = 0.5,
    offspring_range        = (1, 2),
    max_population         = 2000,
    num_runs               = 15,
    initial_altruist_ratio = 0.5,
    initial_green_beard_ratio = 0.5,
    random_seed            = 42,
)

# Variamos el ratio inicial de barba verde para ver cómo afecta
beard_ratios  = [0.1, 0.25, 0.5, 0.75, 0.9]
exp3_variants = []
for gr in beard_ratios:
    cfg  = cfg_exp3_base.with_changes(initial_green_beard_ratio=gr)
    runs = run_multiple_simulations(
        config                = cfg,
        strategy              = Experiment3Strategy(),
        population_factory_fn = PopulationFactory.create_exp3_population,
    )
    exp3_variants.append((f"Barba={gr:.0%}", runs))
    print(f"Ratio barba {gr:.0%} → procesado")

In [ ]:
# ── Evolución del caso central (barba=50%) ────────────────────────────────
_, runs_exp3_central = exp3_variants[beard_ratios.index(0.5)]

plot_population_evolution(
    runs_exp3_central, cfg_exp3_base,
    title      = "Experimento 3 — Evolución de la Población (barba 50%)",
    show_types = "exp3",
)

plot_altruist_fraction(
    runs_exp3_central, cfg_exp3_base,
    title = "Experimento 3 — Fracción de Altruistas (barba 50%)",
)

In [ ]:
plot_exp3_composition_bars(
    configs_and_runs = exp3_variants,
    title = "Experimento 3 — Composición Final según Ratio Inicial de Barba Verde",
)

In [ ]:
compare_escape_probabilities(
    escape_probs          = [0.1, 0.25, 0.5, 0.75, 1.0],
    config_base           = cfg_exp3_base,
    strategy_class        = Experiment3Strategy,
    population_factory_fn = PopulationFactory.create_exp3_population,
    title = "Experimento 3 — Fracción Final de Altruistas vs. Probabilidad de Escape",
)

---
## Conclusiones

### Experimento 1 — Sin barba verde
- La cobardía tiende a dominar con probabilidades de escape moderadas o bajas.
- Existe un umbral crítico de `P` por encima del cual el altruismo puede
  mantenerse o crecer, ya que el beneficio de salvar al compañero supera
  el costo propio.

### Experimento 2 — Barba verde correlacionada con altruismo
- La señal de barba verde permite a los altruistas reconocerse entre sí.
- El altruismo es significativamente más estable que en el Experimento 1,
  pues el altruista solo se sacrifica por otros altruistas.
- La barba verde actúa como un mecanismo de **selección de parentesco**.

### Experimento 3 — Genes independientes
- Sin la correlación perfecta, los cobardes con barba verde explotan la señal.
- El altruismo tiende a declinar porque ya no hay garantía de avisar a
  otro altruista.
- El modelo ilustra por qué las señales honestas de parentesco son evolutivamente
  difíciles de mantener cuando pueden ser "falsificadas" por no-altruistas.

### Reflexión general
Los tres experimentos muestran que el altruismo solo es evolutivamente estable
bajo condiciones específicas: cuando existe una señal fiable (barba verde) que
esté perfectamente correlacionada con el comportamiento altruista. De lo contrario,
la selección natural favorece la cobardía.